# Lab 7: A Helmholtz Coil
## Discretization, Loops, and Vectorization
---

### Learning Objectives
By the end of this lab you will be able to:
1. Discretize a continuous wire into a NumPy array of segments
2. Implement the Biot-Savart sum using explicit loops
3. Identify when loop-based code becomes a performance bottleneck
4. Rewrite spatial loops using NumPy broadcasting (vectorization)
5. Use a computational optimization to solve a problem involving a Helmholtz Coil

---

### Background

The on-axis magnetic field of a single circular loop of radius $R$, carrying current $I$, at a point a distance $z$ from the center along the axis is:

$$B_z(z) = \frac{\mu_0 I R^2}{2(R^2 + z^2)^{3/2}}$$

For **two coaxial loops** separated by distance $d$ (placed at $z = -d/2$ and $z = +d/2$), the total field along the axis is simply the sum:

$$B_{\text{total}}(z) = B_z\!\left(z + \frac{d}{2}\right) + B_z\!\left(z - \frac{d}{2}\right)$$

The **Helmholtz condition** — the separation that gives the most uniform central field — turns out to be exactly $d = R$. Your job in this lab is to *verify* this computationally and then *extend* it to a 2D field map.

---
## PART 1: Discretization and the Loop Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Physical constants
mu0 = 4 * np.pi * 1e-7  # T·m/A
I   = 1.0               # Current in Amperes
R   = 0.5               # Coil radius in meters

### Step 1.1 — Discretize a Single Loop

Fill in the blanks below to create arrays representing a circular loop of wire lying in the $xy$-plane, centered at the origin, with $N$ segments.

**Hint:** The parametric equations for a circle of radius $R$ are $x = R\cos\theta$, $y = R\sin\theta$, $z = 0$. The segment direction vector at angle $\theta$ is tangent to the circle: $d\vec{l} = (-\sin\theta,\, \cos\theta,\, 0)\,R\,d\theta$.

In [ ]:
def make_loop(R, N, z_center=0.0):
    """
    Discretize a circular loop into N segments.
    
    Parameters
    ----------
    R        : float — loop radius (meters)
    N        : int   — number of segments
    z_center : float — z-position of the loop center
    
    Returns
    -------
    x, y, z        : arrays of segment midpoint coordinates, shape (N,)
    dl_x, dl_y, dl_z : arrays of segment direction vectors, shape (N,)
    """
    # Angle values for each segment midpoint
    # IMPORTANT: use endpoint=False so the last point doesn't repeat the first!
    theta  = np.linspace(0, 2*np.pi, N, endpoint=False)
    dtheta = 2 * np.pi / N  # angular step size
    
    # --- FILL IN: segment midpoint positions ---
    x = ___________________   # R * cos(theta)
    y = ___________________   # R * sin(theta)
    z = np.full(N, z_center)  # all segments at the same z height
    
    # --- FILL IN: segment direction vectors (dl) ---
    # These are tangent vectors scaled by the arc length of each segment
    dl_x = ___________________  # -R * sin(theta) * dtheta
    dl_y = ___________________  #  R * cos(theta) * dtheta
    dl_z = np.zeros(N)
    
    return x, y, z, dl_x, dl_y, dl_z


# Test it
N = 200
x, y, z, dl_x, dl_y, dl_z = make_loop(R, N)
print(f"Number of segments: {len(x)}")
print(f"First segment position: ({x[0]:.4f}, {y[0]:.4f}, {z[0]:.4f})")
print(f"Loop circumference check: {np.sum(np.sqrt(dl_x**2 + dl_y**2)):.4f} m  (expect: {2*np.pi*R:.4f} m)")

### Step 1.2 — Compute $B_z$ on the Axis Using a Loop

Fill in the Biot-Savart cross product terms inside the loop below. We are computing the field at points along the $z$-axis, so every observation point has $x_{obs}=0$, $y_{obs}=0$.

In [ ]:
def Bz_axis_loop(x_w, y_w, z_w, dl_x, dl_y, dl_z, z_obs_array, I=1.0):
    """
    Compute Bz along the z-axis for ONE loop using a Python for loop.
    Observation points are all on the z-axis: (0, 0, z_obs).
    
    Returns
    -------
    Bz_vals : array of shape (len(z_obs_array),)
    """
    Bz_vals = np.zeros(len(z_obs_array))
    
    for j, z_obs in enumerate(z_obs_array):     # loop over observation points
        Bz_total = 0.0
        for i in range(len(x_w)):               # loop over wire segments
            # Vector from segment midpoint to observation point
            rx = 0.0   - x_w[i]
            ry = 0.0   - y_w[i]
            rz = z_obs - z_w[i]
            r_mag = np.sqrt(rx**2 + ry**2 + rz**2)
            
            # Biot-Savart: dB = (mu0*I/4pi) * (dl × r) / r^3
            # Cross product: dl × r
            # --- FILL IN the z-component of the cross product ---
            # For a loop in the xy-plane at the origin, Bz = dl_x*ry - dl_y*rx (this is the z-component of dl×r)
            cross_z = ___________________
            
            # Accumulate the z-component of dB
            Bz_total += (mu0 * I / (4 * np.pi)) * cross_z / r_mag**3
        
        Bz_vals[j] = Bz_total
    
    return Bz_vals


# Test: compare to analytical result
z_test = np.linspace(-1.5*R, 1.5*R, 50)

start = time.time()
Bz_numerical = Bz_axis_loop(x, y, z, dl_x, dl_y, dl_z, z_test)
loop_time = time.time() - start

# Analytical on-axis field for a single loop
Bz_analytical = (mu0 * I * R**2) / (2 * (R**2 + z_test**2)**1.5)

print(f"Loop computation time: {loop_time*1000:.1f} ms")
max_error = np.max(np.abs(Bz_numerical - Bz_analytical)) / np.max(Bz_analytical) * 100
print(f"Max relative error vs analytical: {max_error:.4f}%")

plt.figure(figsize=(8, 4))
plt.plot(z_test/R, Bz_analytical * 1e6, 'k-', lw=2, label='Analytical')
plt.plot(z_test/R, Bz_numerical * 1e6, 'r--', lw=1.5, label=f'Numerical (N={N})')
plt.xlabel('z / R')
plt.ylabel('$B_z$ (μT)')
plt.title('On-axis field: single loop')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 1.3 — Two-Coil Superposition and the Helmholtz Configuration

Now build the two-coil system. Place one loop at $z = +d/2$ and one at $z = -d/2$. 

**Complete the function below**, then test it at $d = R$ (the Helmholtz condition) and compare to the analytical formula.

In [ ]:
def Bz_helmholtz_loop(d, z_obs_array, N=200, I=1.0):
    """
    On-axis Bz for two coils separated by distance d.
    Uses the loop-based Biot-Savart from Step 1.2.
    """
    # --- FILL IN: create the top coil at z = +d/2 ---
    x1, y1, z1, dl_x1, dl_y1, dl_z1 = make_loop(R, N, z_center=___)
    
    # --- FILL IN: create the bottom coil at z = -d/2 ---
    x2, y2, z2, dl_x2, dl_y2, dl_z2 = make_loop(R, N, z_center=___)
    
    # Compute Bz from each coil and add them
    Bz1 = Bz_axis_loop(x1, y1, z1, dl_x1, dl_y1, dl_z1, z_obs_array, I)
    Bz2 = Bz_axis_loop(x2, y2, z2, dl_x2, dl_y2, dl_z2, z_obs_array, I)
    
    return Bz1 + Bz2


# Test at Helmholtz condition: d = R
z_axis = np.linspace(-R, R, 200)
d_helmholtz = R

Bz_two_coils = Bz_helmholtz_loop(d_helmholtz, z_axis)

# Analytical superposition
def Bz_single_analytical(z, z_coil):
    return (mu0 * I * R**2) / (2 * (R**2 + (z - z_coil)**2)**1.5)

Bz_analytical_2 = Bz_single_analytical(z_axis, d_helmholtz/2) + Bz_single_analytical(z_axis, -d_helmholtz/2)

plt.figure(figsize=(8, 4))
plt.plot(z_axis/R, Bz_analytical_2 * 1e6, 'k-', lw=2, label='Analytical (two coils)')
plt.plot(z_axis/R, Bz_two_coils * 1e6,   'r--', lw=1.5, label='Numerical (two coils)')
plt.axvspan(-0.1, 0.1, alpha=0.15, color='blue', label='Target zone (±0.1R)')
plt.xlabel('z / R')
plt.ylabel('$B_z$ (μT)')
plt.title(f'Helmholtz coils: d = R = {R} m')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Uniformity check
B_center = Bz_two_coils[len(z_axis)//2]
# Find index closest to z = 0.1*R
idx_edge = np.argmin(np.abs(z_axis - 0.1*R))
B_edge   = Bz_two_coils[idx_edge]
variation = np.abs(B_center - B_edge) / B_center * 100
print(f"Field at center:       {B_center*1e6:.4f} μT")
print(f"Field at z = 0.1R:     {B_edge*1e6:.4f} μT")
print(f"Variation in ±0.1R zone: {variation:.4f}%")

### Step 1.4 — Benchmarking

Run the timing experiment below. You will benchmark the loop-based code at several grid sizes to empirically measure how the runtime scales.

In [ ]:
grid_sizes = [10, 20, 40, 80]
times = []

# Use a single loop for this test
x_s, y_s, z_s, dlx_s, dly_s, dlz_s = make_loop(R, N=100)

for gs in grid_sizes:
    z_test_gs = np.linspace(-R, R, gs)
    t0 = time.time()
    _ = Bz_axis_loop(x_s, y_s, z_s, dlx_s, dly_s, dlz_s, z_test_gs)
    times.append(time.time() - t0)
    print(f"  Grid size {gs:3d}: {times[-1]*1000:.2f} ms")

plt.figure(figsize=(6, 4))
plt.loglog(grid_sizes, times, 'bo-', lw=2, label='Measured')
# Overlay the expected O(M) line for reference (this is 1D, so linear)
ref = np.array(times[0]) * np.array(grid_sizes) / grid_sizes[0]
plt.loglog(grid_sizes, ref, 'k--', alpha=0.5, label='O(M) reference')
plt.xlabel('Number of observation points M')
plt.ylabel('Execution time (s)')
plt.title('Scaling: loop-based Biot-Savart')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nQ: Does the scaling match your prediction? Why or why not?")

### Mini-Tutorial: How Broadcasting Replaces Loops

Before tackling the 3D magnetic field, let's look at a simpler 1D problem. Imagine you have **3 lightbulbs** (sources) and you want to calculate the distance to **4 observation points** (targets). 

Using a loop, you would pick a bulb, measure the distance to all 4 points, then move to the next bulb. 

**Broadcasting** allows us to do this all at once by playing with the "shapes" of the arrays. If we turn the 3 sources into a column and the 4 targets into a row, NumPy will automatically "duplicate" (broadcast) the values to fill a 3x4 grid, calculating every combination simultaneously.

In [ ]:
# Run this cell to see broadcasting in action!

# 1. Define our 1D arrays
sources = np.array([1, 2, 3])          # 3 lightbulbs at x = 1, 2, 3
targets = np.array([10, 20, 30, 40])   # 4 targets at x = 10, 20, 30, 40

print(f"Original sources shape: {sources.shape}")
print(f"Original targets shape: {targets.shape}\n")

# 2. Add 'dummy' axes using np.newaxis
# This changes a flat array into a column or a row
sources_col = sources[:, np.newaxis]   # Becomes a 3x1 column
targets_row = targets[np.newaxis, :]   # Becomes a 1x4 row

print(f"Sources as column shape: {sources_col.shape}")
print(f"Targets as row shape:    {targets_row.shape}\n")

# 3. Calculate all distances at once
# NumPy sees a (3, 1) and a (1, 4). It "stretches" them both into a (3, 4) grid to subtract!
distances = targets_row - sources_col

print(f"Resulting distance matrix shape: {distances.shape}")
print("Distance Matrix:")
print(distances)

# 4. Collapse the sources (summing the contributions)
# If we want the total light hitting each target, we sum down the columns (axis=0)
total_light = np.sum(distances, axis=0)
print(f"\nTotal contribution at each target (shape {total_light.shape}):")
print(total_light)

### Mapping this to the Biot-Savart Law

In the Biot-Savart law, you aren't dealing with 3 bulbs and 4 points. You are dealing with $N$ wire segments and an $M \times M$ grid of observation points. But the logic is exactly the same!

1. **The Wire ($N$ segments):** Instead of `(N,)`, make it a 3D column-like structure: `shape (N, 1, 1)` using `wire_x[:, np.newaxis, np.newaxis]`.
2. **The Grid ($M \times M$ points):** Instead of `(M, M)`, make it a 3D row-like structure: `shape (1, M, M)` using `grid_X[np.newaxis, :, :]`.
3. **The Physics (Math):** When you subtract them (`grid_X - wire_x`), NumPy stretches everything into a massive `(N, M, M)` block, calculating the $x$-distance from *every* wire segment to *every* grid point simultaneously.
4. **The Superposition (Sum):** Just like the lightbulbs, use `np.sum(..., axis=0)` to squash the $N$ dimension, leaving you with the final `(M, M)` grid of magnetic field vectors.

---
## PART 2: Vectorization and Optimization

You now have a working but slow implementation. Your task is to:

1. **Write a vectorized version** of the Biot-Savart sum that eliminates the loop over observation points
2. **Extend it to 2D** — compute the field on an $(x, z)$ grid, not just the axis
3. **Build a uniformity optimizer** to find the best Helmholtz separation distance
4. **Visualize** the result

There is no step-by-step template here. Use the code from lecture and Part 1 as your toolkit.

Guidance (not step-by-step instructions):
- The key broadcasting trick: a wire array of shape `(N,)` becomes `(N, 1, 1)` with `[:, np.newaxis, np.newaxis]`. A 2D grid of shape `(M, M)` becomes `(1, M, M)` with `[np.newaxis, :, :]`. The difference has shape `(N, M, M)`.
- After computing all contributions, `np.sum(..., axis=0)` collapses the wire-segment axis.
- Your uniformity metric should be a single number you can minimize. Think about what 'uniform' means mathematically.
- A grid of 100×100 points in the $(x, z)$ plane from $-1.5R$ to $+1.5R$ is a good starting point.

In [ ]:
# ============================================================
#   Write a vectorized Bz function
#  Inputs:  wire arrays (N,), 2D grid arrays X_obs, Z_obs (M, M)
#  Output:  Bz array (M, M)
# ============================================================

def Bz_vectorized(x_w, y_w, z_w, dl_x, dl_y, dl_z, X_obs, Y_obs, Z_obs, I=1.0):
    """
    YOUR CODE HERE
    """
    pass


# ============================================================
#  Verify your vectorized function against the
#  analytical on-axis formula (at y=0, over a range of z).
#  Print the max relative error — it should be < 0.1%
# ============================================================

# YOUR VERIFICATION CODE HERE


In [ ]:
# ============================================================
#  Benchmark vectorized vs loop
#  Compare timing on a 100x100 grid.
# ============================================================

# YOUR BENCHMARK CODE HERE


In [ ]:
# ============================================================
#   Uniformity optimizer
#  Test 50 values of d from 0.5*R to 1.5*R.
#  For each d, compute your uniformity metric in the central zone.
#  Find and print the optimal d.
#  Plot uniformity metric vs. d — does the minimum land near d=R?
# ============================================================

# YOUR OPTIMIZER CODE HERE


In [ ]:
# ============================================================
#  Final visualization
#  Using your optimal d, produce a 2D color map (imshow or contourf)
#  of |B| in the (x, z) plane.
#  Overlay the positions of the two coils on the plot.
#  Label axes, include a colorbar.
# ============================================================

# YOUR VISUALIZATION CODE HERE


---
## Written Response (Required)

Answer the following in 3–5 sentences each in your report.

**1. In your own words, explain what NumPy broadcasting is doing in your vectorized function. Why does reshaping the wire arrays to shape `(N, 1, 1)` allow you to avoid the loop over observation points?**




**2. What uniformity metric did you choose, and why is it a reasonable measure of field flatness? Could you think of a different metric that might give a different optimal separation?**



**3. Your 2D field map probably shows some artifacts or unexpected features near the coil wire locations. Describe what you see and explain why it happens numerically.**

